In [1]:
import os

cwd = os.getcwd()
print(cwd)

/content


In [3]:
%pip install ultralytics opencv-python-headless moviepy

  Using cached ultralytics-8.4.12-py3-none-any.whl.metadata (38 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached moviepy-2.2.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached scipy-1.17.0-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached polars-1.38.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached imageio-2.37.2-py3-none-any.whl.metadata (9.7 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
  Using cached proglog-0.1.12-py3-none-any.whl.metadata (794 bytes)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached pillow-11.3.0-cp311-cp311-win_amd64.whl.metadata (9.2 kB)
  Usin

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device



In [1]:
import cv2
import torch
from ultralytics import YOLO
import math
import datetime
from moviepy import VideoFileClip, AudioFileClip, TextClip, CompositeVideoClip
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone, timedelta
import os
import time
import sys

from typing import Literal

PROCESSING_MODE: Literal['CPU', 'GPU'] = 'GPU'

# Check if CUDA is available
if PROCESSING_MODE == 'GPU' and torch.cuda.is_available():
    processing_mode = 'cuda'
elif PROCESSING_MODE == 'GPU' and not torch.cuda.is_available():
    print("WARNING: GPU mode requested but CUDA is not available. Falling back to CPU.")
    processing_mode = 'cpu'
else:
    processing_mode = 'cpu'

print(processing_mode)
device = torch.device(processing_mode)

# Initialize YOLO model and move to GPU
model = YOLO('model_weights/best.pt').to(device)

video_title = '[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07'

# Load the video
input_filename = f"/home/arxeno/Documents/TA/Dataset/{video_title}/result.mp4"
video_path = f'{input_filename}'
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if not cap.isOpened():
    print("Error: Cannot open video file!")
    print(f"Check if file exists: {video_path}")
    sys.exit(1)
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps == 0:
        print("Error: Could not read FPS from video")
    else:
        duration = frame_count / fps
        print(f'fps: {fps}')
        print(f'frame_count: {frame_count}')
        print(f'duration: {duration} seconds')

print('test')

# duration = frame_count / fps

def rescale_bbox_to_original(bbox):
    """Rescale bounding box from YOLO size back to original frame size with custom scaling."""
    x_center, y_center, w, h = bbox

    x_min = x_center - (w / 2)
    y_min = y_center - (h / 2)

    return [x_min, y_min, w, h]

# Initialize tracking variables
object_present_per_second = []
segment_bboxes = {}  # Store bounding boxes per segment start time

# Process frames in batches
batch_size = 8  # Set a batch size for processing multiple frames at once

def detect_objects_in_batch(frames, second_start):
    results = model(frames, verbose=False)  # Process the entire batch on the GPU

    # Check if objects are detected in the current batch
    for idx, result in enumerate(results):
        if len(result.boxes) > 0:
            yolo_bbox = result.boxes[0].xywh.cpu().numpy()[0]  # Get first bbox from YOLO

            # Rescale the YOLO bbox back to the original frame size
            candidate_bbox = rescale_bbox_to_original(yolo_bbox)

            # Save the bounding box for this second if not already saved
            current_second = second_start + idx
            if current_second not in segment_bboxes:
                segment_bboxes[current_second] = candidate_bbox

    return [len(result.boxes) > 0 for result in results]

def crop_frame_by_bbox(frame, bbox):
    """Crop the frame based on bounding box coordinates, ensuring w and h are even."""
    x_min, y_min, w, h = [int(coord) for coord in bbox]

    # Ensure coordinates are within image bounds
    x_min = max(0, x_min)
    y_min = max(0, y_min)

    # Adjust w and h to be even
    if w % 2 != 0:
        w -= 1  # Make w even
    if h % 2 != 0:
        h -= 1  # Make h even

    x_max = min(frame.shape[1], x_min + w)  # Ensure we don't go beyond width
    y_max = min(frame.shape[0], y_min + h)  # Ensure we don't go beyond height

    return frame[y_min:y_max, x_min:x_max]

def seconds_to_time(seconds):
    return str(timedelta(seconds=int(seconds)))

second = 0
while second < math.floor(duration):
    print(f'test2 - Processing second {second}')
    frames = []
    timestamps = []
    original_frame_sizes = []

    for i in range(batch_size):
        target_second = second + i
        if target_second >= math.floor(duration):
            break

        print(f'test4 - Reading frame at second {target_second}')
        cap.set(cv2.CAP_PROP_POS_MSEC, target_second * 1000)
        ret, frame = cap.read()

        print(f'  ret={ret}, frame is None: {frame is None}')  # Debug

        if not ret:
            print(f'  Failed to read frame at second {target_second}')
            continue  # Try next frame instead of breaking

        original_frame_sizes.append((frame.shape[1], frame.shape[0]))
        frames.append(frame)
        timestamps.append(target_second)

    if not frames:
        print('No frames to process')
        break

    # Perform batch detection
    batch_results = detect_objects_in_batch(frames, second)
    print(f'batch_results: {batch_results}')

    # Record the results
    for i, result in enumerate(batch_results):
        print('test5')
        object_present_per_second.append(result)

    second += batch_size

cap.release()
print(f"Total seconds scanned: {len(object_present_per_second)}")
print(f"Seconds with objects detected: {sum(object_present_per_second)}")
print(f"Detection timeline: {object_present_per_second[:50]}...")  # First 50 seconds

def split_video(object_present_per_second):
    segments = []
    start = None

    for second, present in enumerate(object_present_per_second):
        if present and start is None:
            start = second
        elif not present and start is not None:
            end = second
            if end - start >= 5:
                segments.append((start, end))
            start = None

    if start is not None and len(object_present_per_second) - start >= 5:
        segments.append((start, len(object_present_per_second)))

    with open("object_detection_times.txt", "w") as f:
        for start_time, end_time in segments:
            start_str = seconds_to_time(start_time)
            end_str = seconds_to_time(end_time)
            f.write(f"{start_str} - {end_str}\n")

    return segments

def process_segment(video_path, start_time, end_time, idx, max_duration):
    print(f"Processing segment {idx}: {start_time} to {end_time}")

    video = VideoFileClip(video_path)
    end_time = min(end_time, max_duration)

    subclip = video.subclipped(start_time, end_time)

    # Get the bounding box for the segment (from the first frame where the object appears)
    first_bbox = segment_bboxes.get(start_time)

    def crop_frame(frame):
        cropped_frame = crop_frame_by_bbox(frame, first_bbox)
        return cropped_frame

    cropped_subclip = subclip.image_transform(crop_frame)

    # Name the file based on start time (use hhmmss format)
    start_str = datetime.fromtimestamp(start_time, timezone.utc).strftime('%H%M%S')
    output_filename = f"{input_filename}_{start_str}.mp4"
    print(f"Output filename: {output_filename}")

    crf_value = 16
    video_codec = "libx264"
    audio_codec = "aac"

    output_filename = f"{output_filename.rsplit('.', 1)[0]}.mp4"

    if not os.path.exists(output_filename):
        cropped_subclip.write_videofile(
            output_filename,
            codec=video_codec,
            audio_codec=audio_codec,
            preset="veryslow",
            threads=4,
            temp_audiofile=f"{output_filename}_temp_audio.m4a",
            remove_temp=True,
            ffmpeg_params=["-crf", str(crf_value)]
        )

video = VideoFileClip(video_path)
max_duration = video.duration
segments = split_video(object_present_per_second)

with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(process_segment, video_path, start_time, end_time, idx, max_duration)
        for idx, (start_time, end_time) in enumerate(segments)
    ]
    print(f'futures length: {len(futures)}')

    for future in futures:
        future.result()

print('test3')

cuda
fps: 24.0
frame_count: 125101
duration: 5212.541666666667 seconds
test
test2 - Processing second 0
test4 - Reading frame at second 0
  ret=True, frame is None: False
test4 - Reading frame at second 1
  ret=True, frame is None: False
test4 - Reading frame at second 2
  ret=True, frame is None: False
test4 - Reading frame at second 3
  ret=True, frame is None: False
test4 - Reading frame at second 4
  ret=True, frame is None: False
test4 - Reading frame at second 5
  ret=True, frame is None: False
test4 - Reading frame at second 6
  ret=True, frame is None: False
test4 - Reading frame at second 7
  ret=True, frame is None: False
batch_results: [False, False, False, False, False, False, False, False]
test5
test5
test5
test5
test5
test5
test5
test5
test2 - Processing second 8
test4 - Reading frame at second 8
  ret=True, frame is None: False
test4 - Reading frame at second 9
  ret=True, frame is None: False
test4 - Reading frame at second 10
  ret=True, frame is None: False
test4 - Re

chunk:   1%|          | 113/20573 [00:00<00:48, 421.27it/s, now=None]

Output filename: /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_005131.mp4
MoviePy - Building video /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_005131.mp4.
MoviePy - Writing audio in /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_005131.mp4_temp_audio.m4a


Output filename: /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_002233.mp4
MoviePy - Building video /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_002233.mp4.
Output filename: /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_003952.mp4



                                                                     
chunk:   1%|          | 113/20573 [00:00<00:48, 421.27it/s, now=None]

...

MoviePy - Done !
MoviePy - video ready /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_003952.mp4


frame_index:  77%|███████▋  | 17245/22392 [08:28<02:17, 37.51it/s, now=None]

frame_index:  78%|███████▊  | 17396/22392 [08:31<01:31, 54.56it/s, now=None]

frame_index:  78%|███████▊  | 17441/22392 [08:32<01:29, 55.56it/s, now=None]

frame_index:  79%|███████▊  | 17588/22392 [08:35<01:31, 52.27it/s, now=None]

frame_index:  80%|███████▉  | 17845/22392 [08:40<01:32, 49.34it/s, now=None]

frame_index:  80%|████████  | 17931/22392 [08:42<01:29, 50.01it/s, now=None]

frame_index:  80%|████████  | 17992/22392 [08:43<01:24, 52.01it/s, now=None]

frame_index:  80%|████████  | 18004/22392 [08:43<01:27, 50.24it/s, now=None]

frame_index:  81%|████████  | 18186/22392 [08:47<01:17, 54.13it/s, now=None]

frame_index:  81%|████████▏ | 18240/22392 [08:48<01:16, 54.24it/s, now=None]

frame_index:  82%|████████▏ | 18275/22392 [08:49<01:15, 54.24it/s, now=None]

frame_index:  82%|████████▏ | 18340/22392 [08:50<01:17, 52.08it/s, now=None]

frame_index:  82%|████████▏ | 18392/22392 [08:51<01:29, 44.62it/

MoviePy - Done !
MoviePy - video ready /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_000319.mp4


MoviePy - Done !
MoviePy - video ready /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_002233.mp4


MoviePy - Done !
MoviePy - video ready /home/arxeno/Documents/TA/Dataset/[FULL] 29 Korban Kapal Tenggelam Masih Hilang iNews Siang 04 07/result.mp4_005131.mp4
test3


In [ ]:
import cv2

# Check OpenCV build info for AV1/dav1d support
build_info = cv2.getBuildInformation()

print("=== FFmpeg support ===")
for line in build_info.split('\n'):
    if any(k in line.lower() for k in ['ffmpeg', 'avcodec', 'avformat', 'dav1d', 'av1']):
        print(line)

# Quick functional test: try opening an AV1 video
print("\n=== Codec test ===")
# Create a VideoCapture with your AV1 file
cap = cv2.VideoCapture("/run/media/arxeno/Volume2/1.KULIAH/SMT 8/1.MATKUL/TA/TA/INews/Dataset/FULL Mbah Tupon di Bantul jadi Korban Mafia Tanah iNews Siang 28 04/h264-converted.mp4")

if cap.isOpened():
    ret, frame = cap.read()
    if ret and frame is not None:
        print(f"SUCCESS: Read frame {frame.shape}")
    else:
        print("FAILED: Opened file but cannot read frames (codec issue)")
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    codec = "".join([chr((fourcc >> 8 * i) & 0xFF) for i in range(4)])
    print(f"Video codec fourcc: {codec}")
    cap.release()
else:
    print("FAILED: Cannot open video file")

=== FFmpeg support ===
    Linker flags (Release):      -Wl,--exclude-libs,libippicv.a -Wl,--exclude-libs,libippiw.a -L/ffmpeg_build/lib  -Wl,--gc-sections -Wl,--as-needed -Wl,--no-undefined  
    Linker flags (Debug):        -Wl,--exclude-libs,libippicv.a -Wl,--exclude-libs,libippiw.a -L/ffmpeg_build/lib  -Wl,--gc-sections -Wl,--as-needed -Wl,--no-undefined  
    FFMPEG:                      YES
      avcodec:                   YES (62.11.100)
      avformat:                  YES (62.3.100)

=== Codec test ===
FAILED: Opened file but cannot read frames (codec issue)
Video codec fourcc: AV01


[av1 @ 0x557c1220c800] Your platform doesn't support hardware accelerated AV1 decoding.
[av1 @ 0x557c1220c800] Failed to get pixel format.
[av1 @ 0x557c1220c800] Get current frame error
